In [1]:
%%capture
!pip install unsloth evaluate

In [2]:
from huggingface_hub import login

hf_token = "hf_****"
login(hf_token)

### Data Prep

In [34]:
%%capture
from datasets import load_dataset, DatasetDict
train_ds = load_dataset("ikram98ai/compliance_verification",split='train')
test_ds = load_dataset("ikram98ai/compliance_verification", split='test')
val_ds = load_dataset("ikram98ai/compliance_verification", split='val')



In [36]:
dataset = DatasetDict({
    'train': train_ds,
    'test': test_ds,
    'validation': val_ds
})
dataset.push_to_hub("johnhmeyer123/compliance_verification_dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/johnhmeyer123/compliance_verification_dataset/commit/7daa54c149525574e58781e4f2d00c737d9ead3a', commit_message='Upload dataset', commit_description='', oid='7daa54c149525574e58781e4f2d00c737d9ead3a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/johnhmeyer123/compliance_verification_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='johnhmeyer123/compliance_verification_dataset'), pr_revision=None, pr_num=None)

In [4]:
system_prompt = """You are a licensing compliance expert specifically for university and Greek organization apparel.
Your task is to evaluate designs against the established licensing guidelines of these specific organizations. Determine
if a design meets all requirements or violates any rules, assuming proper licensing permissions are already in place.
For each evaluation, you must respond in a strict two-line format: first indicating 'Compliance Status: Compliant' or
'Compliance Status: Non-compliant', followed by 'Violation Reason:' with either 'None' for compliant designs or a brief
explanation for non-compliant designs. Never elaborate beyond this format. Base your evaluation solely on actual violations
present in the image, not hypothetical concerns."""

instruction = """Review this apparel design for compliance with licensing rules. Provide compliance status and violation reason, if any."""

In [5]:
import requests
from PIL import Image as PILImage
from io import BytesIO

def load_image_from_url(url):
    """Helper function to download and convert image from URL"""
    try:
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()
        return PILImage.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Error loading image from {url}: {str(e)}")
        return None


In [27]:
%%capture
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "ikram98ai/compliance_verification_lora_model", 
    load_in_4bit = True, 
)

In [28]:
for sample in test_ds.batch(3):
    print(sample)
    break

{'image_urls': [['https://res.cloudinary.com/dsnai3oon/image/upload/v1740658626/cropped_images/5ade41ab877567d2fbffd5f9bbbaee00_1740658626.jpg'], ['https://cf.freshprints.com/designs/1709563046451lqevm_nt_front.png', 'https://cf.freshprints.com/designs/1709563046451breqv_nt_back.png'], ['https://cf.freshprints.com/designs/1708943184370rxcea_nt_front.png', 'https://cf.freshprints.com/designs/1708943184370nfjqr_nt_back.png']], 'compliance_status': ['Non-compliant', 'Compliant', 'Compliant'], 'violation_reason': ['Needs to say "Babson College" fully on the design, recommended next to one of other logos', 'None', 'None']}


In [29]:
import evaluate
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

In [30]:
FastVisionModel.for_inference(model) # Enable for inference!


predictions = []
references = []
for i, batch in enumerate(test_ds.batch(10)):
    if i >2:
        break
    
    print('batch no#: ',i)
    
    preds = []
    for sample in batch['image_urls']:
        message = [
            {"role": "user", "content": [{"type": "image"} for _ in sample]
             +[
                {"type": "text", "text": system_prompt + '\n\n' + instruction}
            ]}
        ]

        input_text = tokenizer.apply_chat_template(message, add_generation_prompt = True)
        inputs = tokenizer(
            [load_image_from_url(img_url) for img_url in sample],
            input_text,
            add_special_tokens = False,
            return_tensors = "pt",
        ).to("cuda")

        generated_text = model.generate(**inputs,max_new_tokens = 128, use_cache = True, temperature = 1.5, min_p = 0.1)
        pred = tokenizer.decode(generated_text[0]).split("Compliance Status: ")[-1].split("\n")[0]
        preds.append(pred)
        
    predictions.extend(preds)
    references.extend(batch['compliance_status'])
 

batch no#:  0
batch no#:  1
batch no#:  2


In [31]:
actuals = [int(str(ref).lower().strip()=="compliant") for ref in references]
preds = [int(str(pred).lower().strip()=="compliant") for pred in predictions]

In [32]:
len(list(zip(references,predictions))), list(zip(references,predictions))

(30,
 [('Non-compliant', 'Non-compliant'),
  ('Compliant', 'Compliant'),
  ('Compliant', 'Non-compliant'),
  ('Compliant', 'Compliant'),
  ('Compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Compliant', 'Compliant'),
  ('Compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Compliant', 'Compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Non-compliant', 'Compliant'),
  ('Compliant', 'Compliant'),
  ('Compliant', 'Non-compliant'),
  ('Compliant', 'Non-compliant'),
  ('Compliant', 'Non-compliant'),
  ('Compliant', 'Compliant'),
  ('Compliant', 'Compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Non-compliant', 'Non-compliant'),
  ('Non-compliant', 'Compliant'),
  ('Non-compliant', 'No

In [33]:

clf_metrics.compute(references=actuals, predictions=preds)

{'accuracy': 0.6666666666666666,
 'f1': 0.5833333333333334,
 'precision': 0.7,
 'recall': 0.5}

In [ ]:
# Qwren's result for 30 test sample
{'accuracy': 0.6666666666666666,
 'f1': 0.5833333333333334,
 'precision': 0.7,
 'recall': 0.5}

# Qwren's result for 110 test sample
{'accuracy': 0.6416666666666667,
 'f1': 0.6194690265486725,
 'precision': 0.625,
 'recall': 0.6140350877192983}

# Llama's result for 110 test sample
{'accuracy': 0.58,
 'f1': 0.6037735849056604,
 'precision': 0.5517241379310345,
 'recall': 0.667}

In [26]:

model.push_to_hub("johnhmeyer123/compliance_verification_lora_model",token= hf_token)
tokenizer.push_to_hub("johnhmeyer123/compliance_verification_lora_model", token= hf_token)

README.md:   0%|          | 0.00/614 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Saved model to https://huggingface.co/johnhmeyer123/compliance_verification_lora_model


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]